树深度4

In [ ]:
#树生成
import pandas as pd
import numpy as np
from itertools import product
import warnings
import os
from pathlib import Path
from tqdm import tqdm
import pickle
from sklearn.linear_model import lars_path
warnings.filterwarnings('ignore')

# ==================== 超参数集中设置 ====================

# 树结构参数
TREE_CONFIG = {
    'data_folder': "1995-2024月度数据",
    'start_year': 1995,
    'end_year': 2024,
    'tree_depth': 4,  # 树深度
    'features': ["规模", "账面市值比", "净资产收益率"],  # 用于划分的特征
    'q_num': 2,  # 分位数划分数（2=中位数划分）
    'ret_col': "超额收益率",  # 收益率列名
    'stock_id_col': "股票代码",  # 股票代码列名
    'month_col': "年月"  # 月份列名
}

# ==================== 树生成部分 ====================

def ntile_series(s, q):
    """与R完全一致的ntile函数"""
    arr = s.values
    out = pd.Series(index=s.index, dtype=float)

    nonnull = s.dropna()
    n = len(nonnull)
    if n == 0:
        return out

    if nonnull.nunique() > 1:
        ranks = nonnull.rank(method="first")
        bins = np.ceil(ranks / (n / q)).astype(int)
        bins[bins > q] = q
        out.loc[nonnull.index] = bins
    else:
        out.loc[nonnull.index] = 1

    return out.astype(int)

def split_node(df, feature, q_num):
    """单层分裂函数，使用中位数划分（二分位数）"""
    df = df.copy()
    df["group"] = ntile_series(df[feature], q_num)
    children = {}

    for g in range(1, q_num + 1):
        mask = df["group"] == g
        children[g] = df.loc[mask].copy()

    return children

def build_tree(df_year, feature_sequence, tree_depth, q_num):
    """
    构建整棵树，使用完整的特征序列
    """
    tree_nodes = {}
    node_id_counter = 1

    # 初始化根节点
    tree_nodes[1] = {
        "node_id": 1,
        "parent": None,
        "depth": 0,
        "feature": None,
        "split_q": None,
        "data": df_year.copy(),
        "children": [],
        "node_name": "root",
        "feature_sequence": feature_sequence
    }

    # BFS 逐层划分
    frontier = [1]

    while frontier:
        node = frontier.pop(0)
        depth = tree_nodes[node]["depth"]

        if depth == tree_depth:
            continue

        split_feature = feature_sequence[depth]
        df_node = tree_nodes[node]["data"]

        if len(df_node) < 2:
            continue

        try:
            children = split_node(df_node, split_feature, q_num)
        except Exception as e:
            print(f"节点 {node} 分裂失败: {e}")
            continue

        for g, df_child in children.items():
            if len(df_child) == 0:
                continue
                
            node_id_counter += 1
            child_id = node_id_counter
            
            if depth == 0:
                node_name = f"root_{split_feature}_q{g}"
            else:
                node_name = f"{tree_nodes[node]['node_name']}_{split_feature}_q{g}"

            tree_nodes[node]["children"].append(child_id)

            tree_nodes[child_id] = {
                "node_id": child_id,
                "parent": node,
                "depth": depth + 1,
                "feature": split_feature,
                "split_q": g,
                "data": df_child,
                "children": [],
                "node_name": node_name,
                "feature_sequence": feature_sequence
            }

            frontier.append(child_id)

    return tree_nodes

def compute_node_return_vector(df_node, config):
    """计算每个节点的12个月等权平均收益向量"""
    ret_col = config['ret_col']
    month_col = config['month_col']
    
    if len(df_node) == 0:
        return np.zeros(12)

    out = np.zeros(12)

    for m in range(1, 13):
        mask = df_node[month_col] == m
        if mask.sum() == 0:
            out[m-1] = 0
            continue

        subset = df_node.loc[mask]
        if len(subset) == 0:
            out[m-1] = 0
            continue

        r = subset[ret_col].values
        
        # 等权平均：投资组合内取平均
        valid_mask = ~np.isnan(r)
        if np.sum(valid_mask) > 0:
            r_valid = r[valid_mask]
            out[m-1] = np.mean(r_valid)
        else:
            out[m-1] = 0

    return out

def build_all_trees_for_one_year(df_year, config):
    """为一年构建所有树"""
    features = config['features']
    tree_depth = config['tree_depth']
    q_num = config['q_num']
    stock_id_col = config['stock_id_col']
    
    feature_sequences = list(product(features, repeat=tree_depth))
    total_sequences = len(feature_sequences)
    print(f"  特征序列数: {total_sequences} ({len(features)}^{tree_depth} = {len(features)**tree_depth})")
    
    all_trees = []
    
    for feat_seq in tqdm(feature_sequences, desc=f"深度{tree_depth}"):
        try:
            tree_nodes = build_tree(df_year, feat_seq, tree_depth=tree_depth, q_num=q_num)

            for nid in tree_nodes:
                df_n = tree_nodes[nid]["data"]
                tree_nodes[nid]["vwret"] = compute_node_return_vector(df_n, config)
                tree_nodes[nid]["stock_count"] = len(df_n[stock_id_col].unique())
                tree_nodes[nid]["observation_count"] = len(df_n)

            all_trees.append({
                "depth": tree_depth,
                "feature_sequence": feat_seq,
                "tree": tree_nodes,
                "total_nodes": len(tree_nodes),
                "leaf_nodes": len([n for n in tree_nodes.values() if not n["children"]])
            })
            
        except Exception as e:
            continue

    success_rate = len(all_trees) / total_sequences * 100
    print(f"  深度 {tree_depth} 完成: {len(all_trees)}/{total_sequences} 棵树 ({success_rate:.1f}%)")
    return all_trees

def load_year_data_simple(config, year):
    """数据加载"""
    data_folder = config['data_folder']
    file_path = Path(data_folder) / f"清洗选中特征后-全市场特征数据_{year}年.csv"
    
    if not file_path.exists():
        raise FileNotFoundError(f"未找到文件: {file_path}")
    
    print(f"  加载文件: {file_path}")
    
    use_cols = [config['month_col'], config['stock_id_col'], config['ret_col'], 
                "规模", "账面市值比", "净资产收益率"]
    df = pd.read_csv(file_path, usecols=use_cols, low_memory=False)
    
    initial_count = len(df)
    df = df.dropna(subset=["规模", "账面市值比", "净资产收益率", config['ret_col']])
    after_count = len(df)
    print(f"  数据清洗: {initial_count} -> {after_count} 行 (删除 {initial_count-after_count} 行缺失值)")
    
    if df[config['month_col']].dtype == 'object':
        if any(df[config['month_col']].str.contains('-', na=False)):
            df[config['month_col']] = pd.to_datetime(df[config['month_col']], errors='coerce').dt.month
        else:
            df[config['month_col']] = pd.to_numeric(df[config['month_col']], errors='coerce')
    
    df = df.dropna(subset=[config['month_col']])
    df[config['month_col']] = df[config['month_col']].astype(int)
    df = df[(df[config['month_col']] >= 1) & (df[config['month_col']] <= 12)]
    
    months = df[config['month_col']].unique()
    print(f"  数据加载完成: {len(df)} 行, {len(months)} 个月份, 股票数: {df[config['stock_id_col']].nunique()}")
    
    return df

def convert_trees_to_ap_pruning_format(all_results, config, output_folder="ap_tree_node_data"):
    """将树结构转换为AP-Pruning所需的格式（节点描述仅保留 node_name 和 depth）"""
    os.makedirs(output_folder, exist_ok=True)
    
    # 收集所有节点信息
    all_nodes_info = []
    all_returns_data = {}
    
    for year, year_results in all_results.items():
        for tree_depth, trees in year_results.items():
            for tree_info in trees:
                tree_nodes = tree_info["tree"]
                
                for node_id, node_data in tree_nodes.items():
                    node_name = node_data["node_name"]
                    depth = node_data["depth"]

                    node_info = {
                        "node_name": node_name,
                        "depth": depth
                    }
                    all_nodes_info.append(node_info)
                    
                    # 收益数据保持不变
                    returns = node_data["vwret"]
                    for month, ret in enumerate(returns, 1):
                        date_key = f"{year}-{month:02d}"
                        if date_key not in all_returns_data:
                            all_returns_data[date_key] = {}
                        all_returns_data[date_key][node_name] = ret
    
    # 节点描述
    nodes_df = pd.DataFrame(all_nodes_info).drop_duplicates()
    nodes_path = Path(output_folder) / "ap_tree_all_nodes_descriptions.csv"
    nodes_df.to_csv(nodes_path, index=False)
    print(f"保存节点描述: {nodes_path} ({len(nodes_df)} 个节点，2 列)")
    
    # 收益数据
    returns_df = pd.DataFrame.from_dict(all_returns_data, orient='index')
    returns_df.index.name = 'date'
    returns_path = Path(output_folder) / "ap_tree_all_nodes_returns.csv"
    returns_df.to_csv(returns_path)
    print(f"保存收益数据: {returns_path} ({len(returns_df)} 期, {len(returns_df.columns)} 个节点)")
    
    return returns_path, nodes_path

def main_tree_creation(config):
    """主函数：创建多树结构"""
    features = config['features']
    q_num = config['q_num']
    start_year = config['start_year']
    end_year = config['end_year']
    tree_depth = config['tree_depth']
    data_folder = config['data_folder']
    
    all_results = {}
    years = list(range(start_year, end_year + 1))
    
    print("=" * 80)
    print("开始构建多树结构")
    print(f"数据期间: {start_year}-{end_year} ({len(years)}年)")
    print(f"树深度: {tree_depth}")
    print(f"划分方法: {q_num}分位数划分")
    print(f"特征: {features}")
    print(f"预期树数量: {len(features)}^{tree_depth} = {len(features)**tree_depth} 棵/年")
    print("=" * 80)
    
    for year in years:
        print(f"\n>>> 处理年份: {year} <<<")
        
        try:
            df_year = load_year_data_simple(config, year)
            
            if len(df_year) == 0:
                print(f"  ! 年份 {year} 无有效数据，跳过")
                continue
            
            monthly_counts = df_year.groupby(config['month_col']).size()
            stock_counts = df_year.groupby(config['month_col'])[config['stock_id_col']].nunique()
            print(f"  月度数据量: {monthly_counts.min()}-{monthly_counts.max()} 行")
            print(f"  月度股票数: {stock_counts.min()}-{stock_counts.max()} 只")
            
            year_trees = build_all_trees_for_one_year(df_year, config)
            all_results[year] = {tree_depth: year_trees}
            
            print(f"  ✓ 年份 {year} 完成:")
            if year_trees:
                total_nodes = sum(tree['total_nodes'] for tree in year_trees)
                total_leaves = sum(tree['leaf_nodes'] for tree in year_trees)
                expected_trees = len(features) ** tree_depth
                print(f"    深度{tree_depth}: {len(year_trees)}/{expected_trees}棵树, 总节点: {total_nodes}, 总叶子: {total_leaves}")
            else:
                print(f"    ! 深度{tree_depth}: 0棵树")
            
        except Exception as e:
            print(f"  ! 处理年份 {year} 时出错: {e}")
            continue
    
    return all_results

def run_tree_generation_only():
    """仅运行树生成的独立函数"""
    config = TREE_CONFIG.copy()
    
    print(" 树生成模式")
    print("当前配置:")
    for key, value in config.items():
        print(f"  {key}: {value}")
    
    # 允许用户修改配置
    tree_depth = input(f"请输入树深度 (当前: {config['tree_depth']}): ").strip()
    if tree_depth:
        config['tree_depth'] = int(tree_depth)
    
    features_input = input(f"请输入特征列表, 用逗号分隔 (当前: {config['features']}): ").strip()
    if features_input:
        config['features'] = [f.strip() for f in features_input.split(',')]
    
    print(f"\n开始构建深度 {config['tree_depth']} 的树结构...")
    print(f"使用特征: {config['features']}")
    
    all_results = main_tree_creation(config)
    
    # # 保存树结构
    # output_folder = f"树结果_深度{config['tree_depth']}"
    # os.makedirs(output_folder, exist_ok=True)
    
    # for year, year_results in all_results.items():
    #     output_path = Path(output_folder) / f"trees_{year}.pkl"
    #     with open(output_path, 'wb') as f:
    #         pickle.dump(year_results, f)
    #     print(f"  ✓ 保存 {year} 年结果: {output_path}")
    
    # 转换为AP-Pruning格式
    print(f"\n转换为AP-Pruning格式...")
    ap_output_folder = f"ap_tree_node_data_深度{config['tree_depth']}"
    returns_path, descriptions_path = convert_trees_to_ap_pruning_format(all_results, config, ap_output_folder)
    
    print(f"\n 树生成完成!")
    #print(f"树结构保存在: {output_folder}")
    print(f"AP-Pruning格式文件:")
    print(f"  - 收益数据: {returns_path}")
    print(f"  - 节点描述: {descriptions_path}")
    
    return all_results, returns_path, descriptions_path

# 运行树生成
if __name__ == "__main__" and 'get_ipython' in globals():
    all_results, returns_path, descriptions_path = run_tree_generation_only()

降维，训练验证测试集未打乱（最原始的版本）

In [ ]:
# Ap_pruning
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.linear_model import lars_path

# ==================== AP-Pruning超参数集中设置 ====================
APPRUNING_CONFIG = {
    'lambda0_list': [0, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.5],      # L1-like parameter used in mu_tilde construction
    'lambda2_list': [0, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.01],    # L2 (ridge) regularization strength used in lars augmentation
    'train_size': 240,                          # 训练集大小（期数）
    'valid_size': 60,                           # 验证集大小（期数）
    'test_size': 60,                            # 测试集大小（期数） - 完全分离 
    'kmin': 5,                                  # 最小节点数
    'kmax': 50,                                 # 最大节点数
    'output_dir': "AP-pruning-output",          # 输出目录
    'depth_weight_power': 2.0,                  # 深度权重指数（用于计算 adj_w）
    'is_excess_return': True,                   # 收益率是否已经是超额收益（减去无风险利率）
    'frequency': 'monthly',                     # 数据频率
    'days_per_month': 30                        # 每月天数假设
}

# ==================== 工具函数 ====================

def lasso_fixed(X, y, lambda2, kmin=5, kmax=50):
    """
    使用 LARS (lasso) 求解路径，lambda2 通过在 X, y 上增加 sqrt(lambda2)*I 的方式实现 ridge-like 正则化。
    """
    n, p = X.shape
    # augment to implement lambda2 (ridge-like)
    yy = np.concatenate([y, np.zeros(p)])
    XX = np.vstack([X, np.sqrt(lambda2) * np.eye(p)])

    try:
        alphas, active, coefs = lars_path(XX, yy, method='lasso')
        beta_path = coefs.T
        K = np.sum(np.abs(beta_path) > 1e-12, axis=1)
        # 筛选在 kmin..kmax 区间内的解
        subset_mask = (K >= kmin) & (K <= kmax)
        if not np.any(subset_mask):
            return np.empty((0, p)), np.array([], dtype=int)
        beta_selected = beta_path[subset_mask]
        K_selected = K[subset_mask].astype(int)
        return beta_selected, K_selected
    except Exception as e:
        print(f"LARS失败: {e}")
        return np.empty((0, p)), np.array([], dtype=int)

def prepare_sigma_mu_tilde_fixed(ports_train, lambda0_list):
    """
    输入:
      ports_train: (T_train x N) numpy array （每列为某节点的时间序列收益）
      lambda0_list: lambda0 值列表
    返回:
      sigma_tilde (N x N) - 调整后的协方差（用于 LARS 的 X）
      mu_tilde (N x len(lambda0_list)) - 每个 lambda0 对应的目标向量
      mu (N,) - 原始节点均值
    """
    mu = np.mean(ports_train, axis=0)
    sigma = np.cov(ports_train, rowvar=False, bias=True)

    # 添加正则化确保数值稳定性
    sigma_reg = sigma + 1e-6 * np.eye(sigma.shape[0])
    
    eigvals, eigvecs = np.linalg.eigh(sigma_reg)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # 使用95%方差解释率选择主成分
    total_variance = np.sum(eigvals)
    explained_variance = np.cumsum(eigvals) / total_variance
    gamma = np.argmax(explained_variance >= 0.95) + 1
    gamma = min(gamma, len(eigvals), ports_train.shape[0], ports_train.shape[1])
    gamma = max(gamma, 1)

    print(f"  特征分解: 保留{gamma}个主成分, 解释方差{explained_variance[gamma-1]:.3f}")

    D = eigvals[:gamma]
    V = eigvecs[:, :gamma]

    sigma_tilde = V @ np.diag(np.sqrt(D)) @ V.T

    mu_tilde_cols = []
    mu_bar = np.mean(mu)
    for lam0 in lambda0_list:
        term = (mu + lam0 * mu_bar).reshape(-1, 1)
        mu_tilde_col = (V @ np.diag(1.0 / np.sqrt(D)) @ V.T @ term).flatten()
        mu_tilde_cols.append(mu_tilde_col)

    mu_tilde = np.column_stack(mu_tilde_cols)
    return sigma_tilde, mu_tilde, mu

def compute_sharpe_ratio_fixed(ports, betas, adj_w):
    """
    计算 Sharpe ratio（改进版）
    - 对每条时序先按 beta 生成组合收益序列，然后去均值再算 Sharpe，
      以避免因 β 绝对幅度导致的偏好（使 CV 更能区分 λ）。
    - ports: (T x N)
    - betas: (M x N) or (N,)  (若传单一 beta，也兼容)
    - adj_w: (N,)
    """
    ports = np.asarray(ports)
    # 标准化 shape
    betas = np.atleast_2d(betas)  # (M, N)

    sharpes = []
    for beta in betas:
        b = beta.copy()
        b = b * adj_w

        # 只做多归一化（中国市场）
        b[b < 0] = 0
        if np.sum(b) > 1e-12:
            w = b / np.sum(b)
        else:
            sharpes.append(0.0)
            continue

        sdf = ports @ w  # (T,)
        

        if len(sdf) > 1 and np.std(sdf, ddof=1) > 1e-12:
            sharpe = np.mean(sdf) / np.std(sdf, ddof=1)
        else:
            sharpe = 0.0
        sharpes.append(sharpe)

    return np.array(sharpes)

def find_best_solution_robust(betas, Ks, train_SR, valid_SR=None, preference="balanced"):
    """
    从给定的 candidate betas 中选一个最优索引。
    规则：
      - 若 valid_SR 可用，优先按 valid_SR 最大；
      - 否则按 train_SR 最大。
      - 若多个候选 SR 相同，优先选 K 小（更稀疏），再以 train_SR 做次要比较。
    返回： (best_beta, best_K, best_train_SR)
    """
    betas = np.atleast_2d(betas)
    Ks = np.asarray(Ks)
    train_SR = np.asarray(train_SR)
    if valid_SR is not None and len(valid_SR) > 0:
        perf = np.asarray(valid_SR)
    else:
        perf = np.asarray(train_SR)

    # 找到最大 perf 的索引集合
    best_val = np.nanmax(perf)
    cand_idxs = np.where(np.isclose(perf, best_val, atol=1e-12))[0]

    if len(cand_idxs) == 1:
        idx = cand_idxs[0]
    else:
        # tie-breaker: 先选最小 K（更稀疏）
        Ks_cand = Ks[cand_idxs]
        minK = np.min(Ks_cand)
        minK_idxs = cand_idxs[np.where(Ks_cand == minK)[0]]
        if len(minK_idxs) == 1:
            idx = minK_idxs[0]
        else:
            # 再用训练集 SR 选最大的
            train_vals = train_SR[minK_idxs]
            idx_sub = np.nanargmax(train_vals)
            idx = minK_idxs[idx_sub]

    return betas[idx], int(Ks[idx]), float(train_SR[idx])

def normalize_weights_china(weights):
    """
    中国市场专用权重归一化：只做多，不允许做空
    """
    weights = np.array(weights).copy()
    weights[weights < 0] = 0 
    if np.sum(weights) < 1e-12:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / np.sum(weights)
    
    return weights

def calculate_realistic_performance(returns, is_excess_return=True, frequency='monthly', days_per_month=30):
    """
    计算真实的性能指标 - 使用直接年化公式
    
    Parameters:
    returns: 收益率序列
    is_excess_return: 是否已经是超额收益（减去无风险利率）
    frequency: 数据频率
    days_per_month: 每月天数假设
    """
    if frequency == 'monthly':
        periods_per_year = 12
    else:
        periods_per_year = 252
    
    # 基础统计
    n_periods = len(returns)
    mean_return = np.mean(returns)
    std_return = np.std(returns, ddof=1)
    
    # 计算总投资收益 (终值-本金)/本金
    total_return = np.prod(1 + returns) - 1
    
    # 使用直接公式计算年化收益率
    # 年化收益率 = 总投资收益 / 本金 / 投资天数 * 365 * 100%
    # 这里总投资收益就是 total_return，本金为1
    investment_days = n_periods * days_per_month
    annual_return = total_return / 5
    
    # 年化波动率仍然使用平方根法则
    annual_volatility = std_return * np.sqrt(periods_per_year)
    
    # 夏普比率计算
    if is_excess_return:
        # 如果已经是超额收益，直接计算
        sharpe_ratio = mean_return / std_return * np.sqrt(periods_per_year) if std_return > 1e-12 else 0.0
    else:
        # 如果不是超额收益，需要减去无风险利率（假设年化3%）
        risk_free_monthly = 0.03 / periods_per_year
        excess_returns = returns - risk_free_monthly
        mean_excess_return = np.mean(excess_returns)
        sharpe_ratio = mean_excess_return / std_return * np.sqrt(periods_per_year) if std_return > 1e-12 else 0.0
    
    # 最大回撤
    cumulative_returns = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cumulative_returns)
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = np.min(drawdown)
    
    # Calmar比率
    if max_drawdown < -1e-12:
        calmar_ratio = annual_return / abs(max_drawdown)
    else:
        calmar_ratio = np.inf
    
    # 胜率
    win_rate = np.mean(returns > 0)
    
    # 偏度和峰度
    from scipy import stats
    skewness = stats.skew(returns) if len(returns) > 2 else 0
    kurtosis = stats.kurtosis(returns) if len(returns) > 2 else 0
    
    return {
        '月度收益率': mean_return,
        '月度波动率': std_return,
        '累计收益率': total_return,
        '投资天数': investment_days,
        '年化收益率': annual_return,
        '年化波动率': annual_volatility,
        '年化夏普比率': sharpe_ratio,
        '最大回撤': max_drawdown,
        'Calmar比率': calmar_ratio,
        '胜率': win_rate,
        '偏度': skewness,
        '峰度': kurtosis,
        '观测期数': n_periods
    }

def check_data_quality(returns):
    """
    检查数据质量，识别可能的计算错误
    """
    print("\n 数据质量检查:")
    print(f"  收益率范围: [{returns.min():.4f}, {returns.max():.4f}]")
    print(f"  收益率均值: {returns.mean():.6f}")
    print(f"  收益率标准差: {returns.std():.6f}")
    print(f"  零收益率比例: {np.mean(np.abs(returns) < 1e-8):.2%}")
    print(f"  极端收益率(>10%): {np.sum(returns > 0.1)}个月")
    print(f"  极端亏损(<-10%): {np.sum(returns < -0.1)}个月")
    
    # 检查是否存在异常值
    Q1 = np.percentile(returns, 25)
    Q3 = np.percentile(returns, 75)
    IQR = Q3 - Q1
    outliers = returns[(returns < Q1 - 1.5*IQR) | (returns > Q3 + 1.5*IQR)]
    print(f"  异常值数量: {len(outliers)}")

# ==================== 主流程：AP-Pruning 辅助与主函数 ====================

def lasso_pruning_helper(ports_train, ports_valid, ports_test, lambda0_list, lambda2_list, 
                         main_dir, adj_w, kmin=5, kmax=50, config=None):
    """
    改进的AP-Pruning辅助函数：避免过拟合，中国市场适配
    """
    if config is None:
        config = {}
    
    output_dir = Path(main_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    sigma_tilde, mu_tilde, mu = prepare_sigma_mu_tilde_fixed(ports_train, lambda0_list)
    N = sigma_tilde.shape[0]

    best_overall_results = []

    print("网格搜索结果（避免选择无正则化的过拟合解）:")
    print("lambda0 | lambda2 | 节点数 | 训练SR | 验证SR")
    print("-" * 55)
    
    for i, lam0 in enumerate(lambda0_list):
        y_i = mu_tilde[:, i]
        for j, lam2 in enumerate(lambda2_list):
            betas, Ks = lasso_fixed(sigma_tilde, y_i, lam2, kmin=kmin, kmax=kmax)
            if len(betas) == 0:
                print(f"  {lam0:7.1f} | {lam2:7.2f} | {'无解':6} | {'--':6} | {'--':6}")
                continue

            train_SR = compute_sharpe_ratio_fixed(ports_train, betas, adj_w)
            valid_SR = compute_sharpe_ratio_fixed(ports_valid, betas, adj_w) if ports_valid is not None else None

            # 选择最佳解
            best_beta, best_K, best_train_SR = find_best_solution_robust(
                betas, Ks, train_SR, valid_SR, preference="regularized"
            )

            best_valid_sr = valid_SR[np.nanargmax(valid_SR)] if valid_SR is not None else None

            result = {
                'lambda0': lam0,
                'lambda2': lam2,
                'train_SR': best_train_SR,
                'valid_SR': best_valid_sr,
                'portsN': int(best_K),
                'beta': best_beta
            }
            best_overall_results.append(result)
            
            print(f"  {lam0:7.1f} | {lam2:7.2f} | {best_K:6d} | {best_train_SR:6.4f} | {best_valid_sr if best_valid_sr is not None else 'N/A':6.4f}")

    if not best_overall_results:
        print(" 未找到有效解")
        return None

    # 选择最终最优解：优先选择有正则化的解,如果没有有正则化的解，才考虑无正则化的
    valid_results = [r for r in best_overall_results if r['lambda2'] > 0]
    if valid_results:
        best_idx = np.argmax([r.get('valid_SR', -np.inf) for r in valid_results])
        best_result = valid_results[best_idx]
        print(f" 选择有正则化的解: lambda0={best_result['lambda0']}, lambda2={best_result['lambda2']}")
    else:
        best_idx = np.argmax([r.get('valid_SR', -np.inf) for r in best_overall_results])
        best_result = best_overall_results[best_idx]
        print(f" 使用无正则化的解，可能过拟合")


    if ports_valid is not None and ports_train is not None:
        ports_trainval = np.vstack([ports_train, ports_valid])
    else:
        ports_trainval = ports_train.copy() if ports_train is not None else None

    try:
        sigma_tilde_tv, mu_tilde_tv, _ = prepare_sigma_mu_tilde_fixed(ports_trainval, [best_result['lambda0']])
        mu_tilde_tv = mu_tilde_tv.flatten()
        betas_tv, Ks_tv = lasso_fixed(sigma_tilde_tv, mu_tilde_tv, best_result['lambda2'], kmin=kmin, kmax=kmax)

        if len(betas_tv) == 0:
            print(" 在 train+valid 上重新训练未返回解，使用原训练集解作为最终解。")
            beta_final = best_result['beta'].copy()
            best_K_final = best_result['portsN']
        else:
            trainval_SR = compute_sharpe_ratio_fixed(ports_trainval, betas_tv, adj_w)
            idx_best_tv = int(np.nanargmax(trainval_SR))
            beta_final = betas_tv[idx_best_tv]
            best_K_final = int(Ks_tv[idx_best_tv])

            print(f" 在 train+valid 上重新训练并选出最终解: K={best_K_final}, idx={idx_best_tv}")
    except Exception as e:
        print(f" 在 train+valid 上重新训练时出错: {e}")
        beta_final = best_result['beta'].copy()
        best_K_final = best_result['portsN']

    # 用最终 beta_final 计算权重并继续后续流程
    beta_final = beta_final.copy()
    b = beta_final * adj_w
    weights = normalize_weights_china(b)
    best_result['weights_sum1'] = weights
    best_result['portsN'] = best_K_final


    # 测试集评估
    if ports_test is not None:
        test_port_returns = ports_test @ weights
        
        check_data_quality(test_port_returns)
        
        is_excess_return = config.get('is_excess_return', True)
        frequency = config.get('frequency', 'monthly')
        days_per_month = config.get('days_per_month', 30)
        
        performance_metrics = calculate_realistic_performance(
            test_port_returns, is_excess_return, frequency, days_per_month
        )
        
        best_result.update(performance_metrics)
        best_result['test_returns_series'] = test_port_returns
        
        print(f"\n 真实测试集性能报告（直接年化公式）:")
        print(f"  累计收益率: {performance_metrics['累计收益率']:.2%}")
        print(f"  投资天数: {performance_metrics['投资天数']}天")
        print(f"  年化收益率: {performance_metrics['年化收益率']:.2%}")
        print(f"  年化波动率: {performance_metrics['年化波动率']:.2%}")
        print(f"  年化夏普比率: {performance_metrics['年化夏普比率']:.3f}")
        print(f"  最大回撤: {performance_metrics['最大回撤']:.2%}")
        print(f"  Calmar比率: {performance_metrics['Calmar比率']:.3f}")
        print(f"  胜率: {performance_metrics['胜率']:.2%}")
        print(f"  偏度: {performance_metrics['偏度']:.3f}")
        print(f"  峰度: {performance_metrics['峰度']:.3f}")
        print(f"  计算方法: 直接年化公式 (年化收益率 = 累计收益 / 投资天数 × 365)")

    out_best = {
        'lambda0': best_result['lambda0'],
        'lambda2': best_result['lambda2'],
        'portsN': best_result['portsN'],
        'train_SR': best_result['train_SR'],
        'valid_SR': best_result.get('valid_SR', np.nan),
        'total_return': best_result.get('累计收益率', np.nan),
        'investment_days': best_result.get('投资天数', np.nan),
        'annual_return': best_result.get('年化收益率', np.nan),
        'annual_volatility': best_result.get('年化波动率', np.nan),
        'annual_sharpe': best_result.get('年化夏普比率', np.nan),
        'max_drawdown': best_result.get('最大回撤', np.nan),
        'calmar_ratio': best_result.get('Calmar比率', np.nan),
        'win_rate': best_result.get('胜率', np.nan),
        'skewness': best_result.get('偏度', np.nan),
        'kurtosis': best_result.get('峰度', np.nan)
    }
    
    results_df = pd.DataFrame([out_best])
    results_df.to_csv(output_dir / 'best_results_realistic.csv', index=False)

    weights_df = pd.DataFrame({
        'node_index': np.arange(len(weights)),
        'weight': weights
    })
    weights_df.to_csv(output_dir / 'selected_weights_indexed.csv', index=False)

    if best_result['test_returns_series'] is not None:
        pd.DataFrame({'test_return': best_result['test_returns_series']}).to_csv(
            output_dir / 'test_returns_series.csv', index=False)

    return best_result

def run_ap_pruning_on_generated_files(returns_path, descriptions_path, config):

    print("开始改进版 AP-Pruning（中国市场适配）...")
    print(f"  returns: {returns_path}")
    print(f"  nodes:   {descriptions_path}")

    returns_df = pd.read_csv(returns_path)
    nodes_df = pd.read_csv(descriptions_path)

    if 'date' not in returns_df.columns:
        raise ValueError("returns file 必须包含 'date' 列")
    
    returns_df['date'] = pd.to_datetime(returns_df['date'], errors='coerce')
    returns_df = returns_df.sort_values('date').reset_index(drop=True)

    node_cols = [c for c in returns_df.columns if c != 'date']
    ports_data = returns_df[node_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    
    total_required = config['train_size'] + config['valid_size'] + config['test_size']
    if len(ports_data) < total_required:
        raise ValueError(f"数据长度不足")

    # 计算深度权重
    depth_map = nodes_df.set_index('node_name')['depth'].to_dict()
    depths = np.array([depth_map.get(col, 0) for col in node_cols], dtype=float)
    adj_w = 1.0 / np.sqrt(config['depth_weight_power'] ** depths)
    adj_w = np.nan_to_num(adj_w, nan=1.0, posinf=1.0, neginf=1.0)

    # 数据划分
    T = len(ports_data)
    train_end = config['train_size']
    valid_end = train_end + config['valid_size']
    test_end = valid_end + config['test_size']

    ports_train = ports_data.iloc[0:train_end].values
    ports_valid = ports_data.iloc[train_end:valid_end].values
    ports_test = ports_data.iloc[valid_end:test_end].values

    print(f"数据划分: train {0}-{train_end-1}, valid {train_end}-{valid_end-1}, test {valid_end}-{test_end-1}")
    print(f"节点数: {len(node_cols)}, 中国市场: 只做多归一化")
    print(f"收益率类型: {'超额收益' if config.get('is_excess_return', True) else '原始收益'}")
    print(f"每月天数假设: {config.get('days_per_month', 30)}天")

    main_dir = Path(config['output_dir'])
    best_result = lasso_pruning_helper(ports_train, ports_valid, ports_test,
                                       config['lambda0_list'], config['lambda2_list'],
                                       main_dir, adj_w, config['kmin'], config['kmax'], config)

    if best_result is not None:
        weights = best_result['weights_sum1']
        weights_named = pd.DataFrame({
            'node_name': node_cols,
            'weight': weights
        }).sort_values('weight', ascending=False).reset_index(drop=True)
        weights_named.to_csv(Path(config['output_dir']) / 'selected_weights_by_node.csv', index=False)
        
        positive_weights = weights[weights > 0]
        print(f"权重统计: 正权重{len(positive_weights)}个, 零权重{len(weights) - len(positive_weights)}个")
        print(f"权重总和: {np.sum(weights):.6f} (应为1.0)")
        
    return config['output_dir'], best_result

def run_ap_pruning_only():
    print(" 改进版 AP-Pruning（中国市场）")
    config = APPRUNING_CONFIG.copy()

    returns_path = input("请输入收益数据文件路径: ").strip()
    descriptions_path = input("请输入节点描述文件路径: ").strip()

    # 确认收益率类型
    is_excess = input("收益率是否已经是超额收益（减去无风险利率）? (y/n, 默认: y): ").strip().lower()
    if is_excess == 'n':
        config['is_excess_return'] = False
        print(" 注意: 将使用原始收益率计算夏普比率")
    else:
        config['is_excess_return'] = True
        print(" 使用超额收益率计算夏普比率")

    # 允许修改每月天数假设
    days_input = input(f"每月天数假设 (默认: {config.get('days_per_month', 30)}): ").strip()
    if days_input:
        config['days_per_month'] = int(days_input)

    if not os.path.exists(returns_path) or not os.path.exists(descriptions_path):
        print("输入文件不存在")
        return None

    output_dir, best_result = run_ap_pruning_on_generated_files(returns_path, descriptions_path, config)
    return output_dir, best_result

if __name__ == "__main__" and 'get_ipython' in globals():
    run_ap_pruning_only()

降维，交叉检验

In [ ]:
# Ap_pruning
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.linear_model import lars_path

# ==================== AP-Pruning超参数集中设置 ====================
APPRUNING_CONFIG = {
    'lambda0_list': [1, 5, 6.5, 7, 7.5, 8, 8.5, 9 ],      # L1-like parameter used in mu_tilde construction
    'lambda2_list': [0.001, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.1, 0.2],    # L2 (ridge) regularization strength used in lars augmentation
    'train_size': 240,                          # 训练集大小（期数）
    'valid_size': 60,                           # 验证集大小（期数）
    'test_size': 60,                            # 测试集大小（期数） - 完全分离 
    'kmin': 5,                                  # 最小节点数
    'kmax': 50,                                 # 最大节点数
    'output_dir': "AP-pruning-output",          # 输出目录
    'depth_weight_power': 2.0,                  # 深度权重指数（用于计算 adj_w）
    'is_excess_return': True,                   # 收益率是否已经是超额收益（减去无风险利率）
    'frequency': 'monthly',                     # 数据频率
    'days_per_month': 30                        # 每月天数假设
}

# ==================== 工具函数 ====================

def lasso_fixed(X, y, lambda2, kmin=5, kmax=50):
    """
    使用 LARS (lasso) 求解路径，lambda2 通过在 X, y 上增加 sqrt(lambda2)*I 的方式实现 ridge-like 正则化。
    """
    n, p = X.shape
    # augment to implement lambda2 (ridge-like)
    yy = np.concatenate([y, np.zeros(p)])
    XX = np.vstack([X, np.sqrt(lambda2) * np.eye(p)])

    try:
        alphas, active, coefs = lars_path(XX, yy, method='lasso')
        beta_path = coefs.T
        K = np.sum(np.abs(beta_path) > 1e-12, axis=1)
        # 筛选在 kmin..kmax 区间内的解
        subset_mask = (K >= kmin) & (K <= kmax)
        if not np.any(subset_mask):
            return np.empty((0, p)), np.array([], dtype=int)
        beta_selected = beta_path[subset_mask]
        K_selected = K[subset_mask].astype(int)
        return beta_selected, K_selected
    except Exception as e:
        print(f"LARS失败: {e}")
        return np.empty((0, p)), np.array([], dtype=int)

def prepare_sigma_mu_tilde_fixed(ports_train, lambda0_list):
    """
    输入:
      ports_train: (T_train x N) numpy array （每列为某节点的时间序列收益）
      lambda0_list: lambda0 值列表
    返回:
      sigma_tilde (N x N) - 调整后的协方差（用于 LARS 的 X）
      mu_tilde (N x len(lambda0_list)) - 每个 lambda0 对应的目标向量
      mu (N,) - 原始节点均值
    """
    mu = np.mean(ports_train, axis=0)
    sigma = np.cov(ports_train, rowvar=False, bias=True)

    # 添加正则化确保数值稳定性
    sigma_reg = sigma + 1e-6 * np.eye(sigma.shape[0])
    
    eigvals, eigvecs = np.linalg.eigh(sigma_reg)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # 使用95%方差解释率选择主成分
    total_variance = np.sum(eigvals)
    explained_variance = np.cumsum(eigvals) / total_variance
    gamma = np.argmax(explained_variance >= 0.95) + 1
    gamma = min(gamma, len(eigvals), ports_train.shape[0], ports_train.shape[1])
    gamma = max(gamma, 1)

    print(f"  特征分解: 保留{gamma}个主成分, 解释方差{explained_variance[gamma-1]:.3f}")

    D = eigvals[:gamma]
    V = eigvecs[:, :gamma]

    sigma_tilde = V @ np.diag(np.sqrt(D)) @ V.T

    mu_tilde_cols = []
    mu_bar = np.mean(mu)
    for lam0 in lambda0_list:
        term = (mu + lam0 * mu_bar).reshape(-1, 1)
        mu_tilde_col = (V @ np.diag(1.0 / np.sqrt(D)) @ V.T @ term).flatten()
        mu_tilde_cols.append(mu_tilde_col)

    mu_tilde = np.column_stack(mu_tilde_cols)
    return sigma_tilde, mu_tilde, mu

def compute_sharpe_ratio_fixed(ports, betas, adj_w):
    """
    计算 Sharpe ratio（改进版）
    - 对每条时序先按 beta 生成组合收益序列，然后去均值再算 Sharpe，
      以避免因 β 绝对幅度导致的偏好（使 CV 更能区分 λ）。
    - ports: (T x N)
    - betas: (M x N) or (N,)  (若传单一 beta，也兼容)
    - adj_w: (N,)
    """
    ports = np.asarray(ports)
    betas = np.atleast_2d(betas)  

    sharpes = []
    for beta in betas:
        b = beta.copy()
        b = b * adj_w

        # 只做多归一化（中国市场）
        b[b < 0] = 0
        if np.sum(b) > 1e-12:
            w = b / np.sum(b)
        else:
            sharpes.append(0.0)
            continue

        sdf = ports @ w  # (T,)
        

        if len(sdf) > 1 and np.std(sdf, ddof=1) > 1e-12:
            sharpe = np.mean(sdf) / np.std(sdf, ddof=1)
        else:
            sharpe = 0.0
        sharpes.append(sharpe)

    return np.array(sharpes)

def find_best_solution_robust(betas, Ks, train_SR, valid_SR=None, preference="balanced"):
    """
    从给定的 candidate betas 中选一个最优索引。
    规则：
      - 若 valid_SR 可用，优先按 valid_SR 最大；
      - 否则按 train_SR 最大。
      - 若多个候选 SR 相同，优先选 K 小（更稀疏），再以 train_SR 做次要比较。
    返回： (best_beta, best_K, best_train_SR)
    """
    betas = np.atleast_2d(betas)
    Ks = np.asarray(Ks)
    train_SR = np.asarray(train_SR)
    if valid_SR is not None and len(valid_SR) > 0:
        perf = np.asarray(valid_SR)
    else:
        perf = np.asarray(train_SR)

    best_val = np.nanmax(perf)
    cand_idxs = np.where(np.isclose(perf, best_val, atol=1e-12))[0]

    if len(cand_idxs) == 1:
        idx = cand_idxs[0]
    else:
        Ks_cand = Ks[cand_idxs]
        minK = np.min(Ks_cand)
        minK_idxs = cand_idxs[np.where(Ks_cand == minK)[0]]
        if len(minK_idxs) == 1:
            idx = minK_idxs[0]
        else:
            train_vals = train_SR[minK_idxs]
            idx_sub = np.nanargmax(train_vals)
            idx = minK_idxs[idx_sub]

    return betas[idx], int(Ks[idx]), float(train_SR[idx])

def normalize_weights_china(weights):
    """
    中国市场专用权重归一化：只做多，不允许做空
    """
    weights = np.array(weights).copy()
    weights[weights < 0] = 0
    if np.sum(weights) < 1e-12:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / np.sum(weights)
    
    return weights

def calculate_realistic_performance(returns, is_excess_return=True, frequency='monthly', days_per_month=30):
    """
    计算真实的性能指标 - 使用直接年化公式
    
    Parameters:
    returns: 收益率序列
    is_excess_return: 是否已经是超额收益（减去无风险利率）
    frequency: 数据频率
    days_per_month: 每月天数假设
    """
    if frequency == 'monthly':
        periods_per_year = 12
    else:
        periods_per_year = 252
    
    # 基础统计
    n_periods = len(returns)
    mean_return = np.mean(returns)
    std_return = np.std(returns, ddof=1)
    
    # 计算总投资收益 (终值-本金)/本金
    total_return = np.prod(1 + returns) - 1
    
    # 使用直接公式计算年化收益率
    # 年化收益率 = 总投资收益 / 本金 / 投资天数 * 365 * 100%
    # 这里总投资收益就是 total_return，本金为1
    investment_days = n_periods * days_per_month
    annual_return = total_return / 5
    
    # 年化波动率仍然使用平方根法则
    annual_volatility = std_return * np.sqrt(periods_per_year)
    
    # 夏普比率计算
    if is_excess_return:
        # 如果已经是超额收益，直接计算
        sharpe_ratio = mean_return / std_return * np.sqrt(periods_per_year) if std_return > 1e-12 else 0.0
    else:
        # 如果不是超额收益，需要减去无风险利率（假设年化3%）
        risk_free_monthly = 0.03 / periods_per_year
        excess_returns = returns - risk_free_monthly
        mean_excess_return = np.mean(excess_returns)
        sharpe_ratio = mean_excess_return / std_return * np.sqrt(periods_per_year) if std_return > 1e-12 else 0.0
    
    # 最大回撤
    cumulative_returns = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cumulative_returns)
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = np.min(drawdown)
    
    # Calmar比率
    if max_drawdown < -1e-12:
        calmar_ratio = annual_return / abs(max_drawdown)
    else:
        calmar_ratio = np.inf
    
    # 胜率
    win_rate = np.mean(returns > 0)
    
    # 偏度和峰度
    from scipy import stats
    skewness = stats.skew(returns) if len(returns) > 2 else 0
    kurtosis = stats.kurtosis(returns) if len(returns) > 2 else 0
    
    return {
        '月度收益率': mean_return,
        '月度波动率': std_return,
        '累计收益率': total_return,
        '投资天数': investment_days,
        '年化收益率': annual_return,
        '年化波动率': annual_volatility,
        '年化夏普比率': sharpe_ratio,
        '最大回撤': max_drawdown,
        'Calmar比率': calmar_ratio,
        '胜率': win_rate,
        '偏度': skewness,
        '峰度': kurtosis,
        '观测期数': n_periods
    }

def check_data_quality(returns):
    """
    检查数据质量，识别可能的计算错误
    """
    print("\n 数据质量检查:")
    print(f"  收益率范围: [{returns.min():.4f}, {returns.max():.4f}]")
    print(f"  收益率均值: {returns.mean():.6f}")
    print(f"  收益率标准差: {returns.std():.6f}")
    print(f"  零收益率比例: {np.mean(np.abs(returns) < 1e-8):.2%}")
    print(f"  极端收益率(>10%): {np.sum(returns > 0.1)}个月")
    print(f"  极端亏损(<-10%): {np.sum(returns < -0.1)}个月")
    
    # 检查是否存在异常值
    Q1 = np.percentile(returns, 25)
    Q3 = np.percentile(returns, 75)
    IQR = Q3 - Q1
    outliers = returns[(returns < Q1 - 1.5*IQR) | (returns > Q3 + 1.5*IQR)]
    print(f"  异常值数量: {len(outliers)}")

# ==================== 主流程：AP-Pruning 辅助与主函数 ====================

def lasso_pruning_helper(ports_trainval, ports_test, lambda0_list, lambda2_list, 
                         main_dir, adj_w, kmin=5, kmax=50, config=None):
    if config is None:
        config = {}
    
    output_dir = Path(main_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    n_samples = ports_trainval.shape[0]
    n_splits = 5
    fold_size = int(n_samples / n_splits)
    

    import sys
    class HiddenPrints:
        def __enter__(self):
            self._original_stdout = sys.stdout
            sys.stdout = open(os.devnull, 'w')
        def __exit__(self, exc_type, exc_val, exc_tb):
            sys.stdout.close()
            sys.stdout = self._original_stdout

    print("网格搜索结果（5折交叉验证平均值）:")
    print("lambda0 | lambda2 | 节点数 | 训练SR | 验证SR")
    print("-" * 55)
    
    grid_results = []

    #网格搜索
    for lam0 in lambda0_list:
        for lam2 in lambda2_list:
            
            cv_valid_srs = []
            cv_train_srs = []
            cv_k_list = []

            # 5折循环
            for fold_idx in range(n_splits):
                start_idx = fold_idx * fold_size
                end_idx = (fold_idx + 1) * fold_size
                
                valid_indices = np.arange(start_idx, end_idx)
                train_indices = np.concatenate([np.arange(0, start_idx), np.arange(end_idx, n_samples)])
                
                X_fold_train = ports_trainval[train_indices]
                X_fold_valid = ports_trainval[valid_indices]

                with HiddenPrints():
                    sigma_tilde, mu_tilde_all, _ = prepare_sigma_mu_tilde_fixed(X_fold_train, [lam0])
                
                y_fold = mu_tilde_all.flatten() 


                betas, Ks = lasso_fixed(sigma_tilde, y_fold, lam2, kmin=kmin, kmax=kmax)
                
                if len(betas) == 0:
                    continue

                fold_train_sr_curve = compute_sharpe_ratio_fixed(X_fold_train, betas, adj_w)
                fold_valid_sr_curve = compute_sharpe_ratio_fixed(X_fold_valid, betas, adj_w)

                idx_best = np.argmax(fold_train_sr_curve)
                
                cv_train_srs.append(fold_train_sr_curve[idx_best])
                cv_valid_srs.append(fold_valid_sr_curve[idx_best])
                cv_k_list.append(Ks[idx_best])

            if len(cv_valid_srs) > 0:
                avg_valid_sr = np.mean(cv_valid_srs)
                avg_train_sr = np.mean(cv_train_srs)
                avg_k = int(np.round(np.mean(cv_k_list)))
                
                grid_results.append({
                    'lambda0': lam0,
                    'lambda2': lam2,
                    'avg_valid_sr': avg_valid_sr,
                    'avg_train_sr': avg_train_sr,
                    'avg_k': avg_k
                })
                print(f"  {lam0:7.1f} | {lam2:7.2f} | {avg_k:6d} | {avg_train_sr:6.4f} | {avg_valid_sr:6.4f}")
            else:
                print(f"  {lam0:7.1f} | {lam2:7.2f} | {'无解':6} | {'--':6} | {'--':6}")

    if not grid_results:
        print("未找到有效解")
        return None

    # 优先选择 lambda2 > 0 的解（正则化），如果都没有才选 lambda2=0
    reg_results = [r for r in grid_results if r['lambda2'] > 0]
    if reg_results:
        best_result = max(reg_results, key=lambda x: x['avg_valid_sr'])
        print(f"选择有正则化的解: lambda0={best_result['lambda0']}, lambda2={best_result['lambda2']}")
    else:
        best_result = max(grid_results, key=lambda x: x['avg_valid_sr'])
        print(f"使用无正则化的解: lambda0={best_result['lambda0']}, lambda2={best_result['lambda2']}")

    sigma_final, mu_tilde_final, _ = prepare_sigma_mu_tilde_fixed(ports_trainval, [best_result['lambda0']])
    y_final = mu_tilde_final.flatten()
    
    betas_final, Ks_final = lasso_fixed(sigma_final, y_final, best_result['lambda2'], kmin=kmin, kmax=kmax)
    
    if len(betas_final) > 0:
        tv_sr_curve = compute_sharpe_ratio_fixed(ports_trainval, betas_final, adj_w)
        final_idx = np.argmax(tv_sr_curve)
        beta_final = betas_final[final_idx]
        best_K_final = int(Ks_final[final_idx])
        print(f"在 train+valid 上重新训练并选出最终解: K={best_K_final}, idx={final_idx}")
    else:
        print("全量重训练无解，无法生成策略")
        return None

    # 计算权重
    b = beta_final * adj_w
    weights = normalize_weights_china(b)
    
    best_result['weights_sum1'] = weights
    best_result['portsN'] = best_K_final
    best_result['beta'] = beta_final
    best_result['train_SR'] = tv_sr_curve[final_idx]
    best_result['valid_SR'] = best_result['avg_valid_sr']

    if ports_test is not None:
        test_port_returns = ports_test @ weights
        
        check_data_quality(test_port_returns)
        
        performance_metrics = calculate_realistic_performance(
            test_port_returns, 
            config.get('is_excess_return', True), 
            config.get('frequency', 'monthly'), 
            config.get('days_per_month', 30)
        )
        
        best_result.update(performance_metrics)
        best_result['test_returns_series'] = test_port_returns
        
        print(f"\n真实测试集性能报告（直接年化公式）:")
        print(f"  累计收益率: {performance_metrics['累计收益率']:.2%}")
        print(f"  投资天数: {performance_metrics['投资天数']}天")
        print(f"  年化收益率: {performance_metrics['年化收益率']:.2%}")
        print(f"  年化波动率: {performance_metrics['年化波动率']:.2%}")
        print(f"  年化夏普比率: {performance_metrics['年化夏普比率']:.3f}")
        print(f"  最大回撤: {performance_metrics['最大回撤']:.2%}")
        print(f"  Calmar比率: {performance_metrics['Calmar比率']:.3f}")
        print(f"  胜率: {performance_metrics['胜率']:.2%}")
        print(f"  偏度: {performance_metrics['偏度']:.3f}")
        print(f"  峰度: {performance_metrics['峰度']:.3f}")
        print(f"  计算方法: 直接年化公式 (年化收益率 = 累计收益 / 投资天数 × 365)")

    # 保存结果
    out_best = {k:v for k,v in best_result.items() if k not in ['weights_sum1', 'beta', 'test_returns_series']}
    pd.DataFrame([out_best]).to_csv(output_dir / 'best_results_realistic.csv', index=False)

    weights_df = pd.DataFrame({'node_index': np.arange(len(weights)), 'weight': weights})
    weights_df.to_csv(output_dir / 'selected_weights_indexed.csv', index=False)

    if best_result['test_returns_series'] is not None:
        pd.DataFrame({'test_return': best_result['test_returns_series']}).to_csv(
            output_dir / 'test_returns_series.csv', index=False)

    return best_result

def run_ap_pruning_on_generated_files(returns_path, descriptions_path, config):

    print("开始改进版 AP-Pruning（中国市场适配）...")
    print(f"  returns: {returns_path}")
    print(f"  nodes:   {descriptions_path}")

    returns_df = pd.read_csv(returns_path)
    nodes_df = pd.read_csv(descriptions_path)

    if 'date' not in returns_df.columns:
        raise ValueError("returns file 必须包含 'date' 列")
    
    returns_df['date'] = pd.to_datetime(returns_df['date'], errors='coerce')
    returns_df = returns_df.sort_values('date').reset_index(drop=True)

    node_cols = [c for c in returns_df.columns if c != 'date']
    ports_data = returns_df[node_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    
    cv_size = config['train_size'] + config['valid_size']
    test_size = config['test_size']
    total_required = cv_size + test_size
    
    if len(ports_data) < total_required:
        raise ValueError(f"数据长度不足: 需要 {total_required}, 实际 {len(ports_data)}")

    # 计算深度权重
    depth_map = nodes_df.set_index('node_name')['depth'].to_dict()
    depths = np.array([depth_map.get(col, 0) for col in node_cols], dtype=float)
    adj_w = 1.0 / np.sqrt(config['depth_weight_power'] ** depths)
    adj_w = np.nan_to_num(adj_w, nan=1.0, posinf=1.0, neginf=1.0)

    # 划分数据
    ports_trainval = ports_data.iloc[0:cv_size].values
    ports_test = ports_data.iloc[cv_size:cv_size+test_size].values

    print(f"Block 划分: 训练+验证共 {cv_size} 期 (用于5折CV), 测试块 {test_size} 期")
    print(f"最终行数: train+valid {cv_size}, test {test_size}")
    print(f"节点数: {len(node_cols)}, 中国市场: 只做多归一化")
    print(f"收益率类型: {'超额收益' if config.get('is_excess_return', True) else '原始收益'}")
    print(f"每月天数假设: {config.get('days_per_month', 30)}天")

    # 运行带CV的AP-Pruning
    main_dir = Path(config['output_dir'])
    best_result = lasso_pruning_helper(ports_trainval, ports_test,
                                       config['lambda0_list'], config['lambda2_list'],
                                       main_dir, adj_w, config['kmin'], config['kmax'], config)

    if best_result is not None:
        weights = best_result['weights_sum1']
        weights_named = pd.DataFrame({
            'node_name': node_cols,
            'weight': weights
        }).sort_values('weight', ascending=False).reset_index(drop=True)
        weights_named.to_csv(Path(config['output_dir']) / 'selected_weights_by_node.csv', index=False)
        
        positive_weights = weights[weights > 0]
        print(f"权重统计: 正权重{len(positive_weights)}个, 零权重{len(weights) - len(positive_weights)}个")
        print(f"权重总和: {np.sum(weights):.6f} (应为1.0)")
        
    return config['output_dir'], best_result

def run_ap_pruning_only():
    print("改进版 AP-Pruning（中国市场）+ 5折交叉检验")
    config = APPRUNING_CONFIG.copy()

    returns_path = input("请输入收益数据文件路径: ").strip()
    descriptions_path = input("请输入节点描述文件路径: ").strip()

    # 确认收益率类型
    is_excess = input("收益率是否已经是超额收益（减去无风险利率）? (y/n, 默认: y): ").strip().lower()
    if is_excess == 'n':
        config['is_excess_return'] = False
        print("注意: 将使用原始收益率计算夏普比率")
    else:
        config['is_excess_return'] = True
        print("使用超额收益率计算夏普比率")

    # 允许修改每月天数假设
    days_input = input(f"每月天数假设 (默认: {config.get('days_per_month', 30)}): ").strip()
    if days_input:
        config['days_per_month'] = int(days_input)

    if not os.path.exists(returns_path) or not os.path.exists(descriptions_path):
        print("输入文件不存在")
        return None

    output_dir, best_result = run_ap_pruning_on_generated_files(returns_path, descriptions_path, config)
    return output_dir, best_result

if __name__ == "__main__" and 'get_ipython' in globals():
    run_ap_pruning_only()

降维，只按照5期分4期给训练集，1期给验证集的方式打乱，不影响测试集

In [ ]:
# Ap_pruning
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.linear_model import lars_path

# ==================== AP-Pruning超参数集中设置 ====================
APPRUNING_CONFIG = {
    'lambda0_list': [0.5, 1, 3, 4, 4.5, 5, 5.5, 6, 6.5, 7],      # L1-like parameter used in mu_tilde construction
    'lambda2_list': [0.00005, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.1],    # L2 (ridge) regularization strength used in lars augmentation
    'train_size': 240,                          # 训练集大小（期数）
    'valid_size': 60,                           # 验证集大小（期数）
    'test_size': 60,                            # 测试集大小（期数） - 完全分离 
    'kmin': 5,                                  # 最小节点数
    'kmax': 50,                                 # 最大节点数
    'output_dir': "AP-pruning-output",          # 输出目录
    'depth_weight_power': 2.0,                  # 深度权重指数（用于计算 adj_w）
    'is_excess_return': True,                   # 收益率是否已经是超额收益（减去无风险利率）
    'frequency': 'monthly',                     # 数据频率
    'days_per_month': 30                        # 每月天数假设
}

# ==================== 工具函数 ====================

def lasso_fixed(X, y, lambda2, kmin=5, kmax=50):
    """
    使用 LARS (lasso) 求解路径，lambda2 通过在 X, y 上增加 sqrt(lambda2)*I 的方式实现 ridge-like 正则化。
    """
    n, p = X.shape
    # augment to implement lambda2 (ridge-like)
    yy = np.concatenate([y, np.zeros(p)])
    XX = np.vstack([X, np.sqrt(lambda2) * np.eye(p)])

    try:
        alphas, active, coefs = lars_path(XX, yy, method='lasso')
        beta_path = coefs.T
        K = np.sum(np.abs(beta_path) > 1e-12, axis=1)
        # 筛选在 kmin..kmax 区间内的解
        subset_mask = (K >= kmin) & (K <= kmax)
        if not np.any(subset_mask):
            return np.empty((0, p)), np.array([], dtype=int)
        beta_selected = beta_path[subset_mask]
        K_selected = K[subset_mask].astype(int)
        return beta_selected, K_selected
    except Exception as e:
        print(f"LARS失败: {e}")
        return np.empty((0, p)), np.array([], dtype=int)

def prepare_sigma_mu_tilde_fixed(ports_train, lambda0_list):
    """
    输入:
      ports_train: (T_train x N) numpy array （每列为某节点的时间序列收益）
      lambda0_list: lambda0 值列表
    返回:
      sigma_tilde (N x N) - 调整后的协方差（用于 LARS 的 X）
      mu_tilde (N x len(lambda0_list)) - 每个 lambda0 对应的目标向量
      mu (N,) - 原始节点均值
    """
    mu = np.mean(ports_train, axis=0)
    sigma = np.cov(ports_train, rowvar=False, bias=True)

    # 添加正则化确保数值稳定性
    sigma_reg = sigma + 1e-6 * np.eye(sigma.shape[0])
    
    eigvals, eigvecs = np.linalg.eigh(sigma_reg)
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    # 使用95%方差解释率选择主成分
    total_variance = np.sum(eigvals)
    explained_variance = np.cumsum(eigvals) / total_variance
    gamma = np.argmax(explained_variance >= 0.95) + 1
    gamma = min(gamma, len(eigvals), ports_train.shape[0], ports_train.shape[1])
    gamma = max(gamma, 1)

    print(f"  特征分解: 保留{gamma}个主成分, 解释方差{explained_variance[gamma-1]:.3f}")

    D = eigvals[:gamma]
    V = eigvecs[:, :gamma]

    sigma_tilde = V @ np.diag(np.sqrt(D)) @ V.T

    mu_tilde_cols = []
    mu_bar = np.mean(mu)
    for lam0 in lambda0_list:
        term = (mu + lam0 * mu_bar).reshape(-1, 1)
        mu_tilde_col = (V @ np.diag(1.0 / np.sqrt(D)) @ V.T @ term).flatten()
        mu_tilde_cols.append(mu_tilde_col)

    mu_tilde = np.column_stack(mu_tilde_cols)
    return sigma_tilde, mu_tilde, mu

def compute_sharpe_ratio_fixed(ports, betas, adj_w):
    """
    计算 Sharpe ratio（改进版）
    - 对每条时序先按 beta 生成组合收益序列，然后去均值再算 Sharpe，
      以避免因 β 绝对幅度导致的偏好（使 CV 更能区分 λ）。
    - ports: (T x N)
    - betas: (M x N) or (N,)  (若传单一 beta，也兼容)
    - adj_w: (N,)
    """
    ports = np.asarray(ports)
    betas = np.atleast_2d(betas)  # (M, N)

    sharpes = []
    for beta in betas:
        b = beta.copy()
        b = b * adj_w

        # 只做多归一化（中国市场）
        b[b < 0] = 0
        if np.sum(b) > 1e-12:
            w = b / np.sum(b)
        else:
            sharpes.append(0.0)
            continue

        sdf = ports @ w  # (T,)
        

        if len(sdf) > 1 and np.std(sdf, ddof=1) > 1e-12:
            sharpe = np.mean(sdf) / np.std(sdf, ddof=1)
        else:
            sharpe = 0.0
        sharpes.append(sharpe)

    return np.array(sharpes)

def find_best_solution_robust(betas, Ks, train_SR, valid_SR=None, preference="balanced"):
    """
    从给定的 candidate betas 中选一个最优索引。
    规则：
      - 若 valid_SR 可用，优先按 valid_SR 最大；
      - 否则按 train_SR 最大。
      - 若多个候选 SR 相同，优先选 K 小（更稀疏），再以 train_SR 做次要比较。
    返回： (best_beta, best_K, best_train_SR)
    """
    betas = np.atleast_2d(betas)
    Ks = np.asarray(Ks)
    train_SR = np.asarray(train_SR)
    if valid_SR is not None and len(valid_SR) > 0:
        perf = np.asarray(valid_SR)
    else:
        perf = np.asarray(train_SR)

    best_val = np.nanmax(perf)
    cand_idxs = np.where(np.isclose(perf, best_val, atol=1e-12))[0]

    if len(cand_idxs) == 1:
        idx = cand_idxs[0]
    else:
        Ks_cand = Ks[cand_idxs]
        minK = np.min(Ks_cand)
        minK_idxs = cand_idxs[np.where(Ks_cand == minK)[0]]
        if len(minK_idxs) == 1:
            idx = minK_idxs[0]
        else:
            train_vals = train_SR[minK_idxs]
            idx_sub = np.nanargmax(train_vals)
            idx = minK_idxs[idx_sub]

    return betas[idx], int(Ks[idx]), float(train_SR[idx])

def normalize_weights_china(weights):
    """
    中国市场专用权重归一化：只做多，不允许做空
    """
    weights = np.array(weights).copy()
    weights[weights < 0] = 0
    if np.sum(weights) < 1e-12:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / np.sum(weights)
    
    return weights

def calculate_realistic_performance(returns, is_excess_return=True, frequency='monthly', days_per_month=30):
    """
    计算真实的性能指标 - 使用直接年化公式
    
    Parameters:
    returns: 收益率序列
    is_excess_return: 是否已经是超额收益（减去无风险利率）
    frequency: 数据频率
    days_per_month: 每月天数假设
    """
    if frequency == 'monthly':
        periods_per_year = 12
    else:
        periods_per_year = 252
    
    # 基础统计
    n_periods = len(returns)
    mean_return = np.mean(returns)
    std_return = np.std(returns, ddof=1)
    
    # 计算总投资收益 (终值-本金)/本金
    total_return = np.prod(1 + returns) - 1
    
    # 使用直接公式计算年化收益率
    # 年化收益率 = 总投资收益 / 本金 / 投资天数 * 365 * 100%
    # 这里总投资收益就是 total_return，本金为1
    investment_days = n_periods * days_per_month
    annual_return = total_return / 5
    
    # 年化波动率仍然使用平方根法则
    annual_volatility = std_return * np.sqrt(periods_per_year)
    
    # 夏普比率计算
    if is_excess_return:
        # 如果已经是超额收益，直接计算
        sharpe_ratio = mean_return / std_return * np.sqrt(periods_per_year) if std_return > 1e-12 else 0.0
    else:
        # 如果不是超额收益，需要减去无风险利率（假设年化3%）
        risk_free_monthly = 0.03 / periods_per_year
        excess_returns = returns - risk_free_monthly
        mean_excess_return = np.mean(excess_returns)
        sharpe_ratio = mean_excess_return / std_return * np.sqrt(periods_per_year) if std_return > 1e-12 else 0.0
    
    # 最大回撤
    cumulative_returns = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cumulative_returns)
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = np.min(drawdown)
    
    # Calmar比率
    if max_drawdown < -1e-12:
        calmar_ratio = annual_return / abs(max_drawdown)
    else:
        calmar_ratio = np.inf
    
    # 胜率
    win_rate = np.mean(returns > 0)
    
    # 偏度和峰度
    from scipy import stats
    skewness = stats.skew(returns) if len(returns) > 2 else 0
    kurtosis = stats.kurtosis(returns) if len(returns) > 2 else 0
    
    return {
        '月度收益率': mean_return,
        '月度波动率': std_return,
        '累计收益率': total_return,
        '投资天数': investment_days,
        '年化收益率': annual_return,
        '年化波动率': annual_volatility,
        '年化夏普比率': sharpe_ratio,
        '最大回撤': max_drawdown,
        'Calmar比率': calmar_ratio,
        '胜率': win_rate,
        '偏度': skewness,
        '峰度': kurtosis,
        '观测期数': n_periods
    }

def check_data_quality(returns):
    """
    检查数据质量，识别可能的计算错误
    """
    print("\n 数据质量检查:")
    print(f"  收益率范围: [{returns.min():.4f}, {returns.max():.4f}]")
    print(f"  收益率均值: {returns.mean():.6f}")
    print(f"  收益率标准差: {returns.std():.6f}")
    print(f"  零收益率比例: {np.mean(np.abs(returns) < 1e-8):.2%}")
    print(f"  极端收益率(>10%): {np.sum(returns > 0.1)}个月")
    print(f"  极端亏损(<-10%): {np.sum(returns < -0.1)}个月")
    
    # 检查是否存在异常值
    Q1 = np.percentile(returns, 25)
    Q3 = np.percentile(returns, 75)
    IQR = Q3 - Q1
    outliers = returns[(returns < Q1 - 1.5*IQR) | (returns > Q3 + 1.5*IQR)]
    print(f"  异常值数量: {len(outliers)}")

# ==================== 主流程：AP-Pruning 辅助与主函数 ====================

def lasso_pruning_helper(ports_train, ports_valid, ports_test, lambda0_list, lambda2_list, 
                         main_dir, adj_w, kmin=5, kmax=50, config=None):
    if config is None:
        config = {}
    
    output_dir = Path(main_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    sigma_tilde, mu_tilde, mu = prepare_sigma_mu_tilde_fixed(ports_train, lambda0_list)
    N = sigma_tilde.shape[0]

    best_overall_results = []

    print("网格搜索结果（避免选择无正则化的过拟合解）:")
    print("lambda0 | lambda2 | 节点数 | 训练SR | 验证SR")
    print("-" * 55)
    
    for i, lam0 in enumerate(lambda0_list):
        y_i = mu_tilde[:, i]
        for j, lam2 in enumerate(lambda2_list):
            betas, Ks = lasso_fixed(sigma_tilde, y_i, lam2, kmin=kmin, kmax=kmax)
            if len(betas) == 0:
                print(f"  {lam0:7.1f} | {lam2:7.2f} | {'无解':6} | {'--':6} | {'--':6}")
                continue

            train_SR = compute_sharpe_ratio_fixed(ports_train, betas, adj_w)
            valid_SR = compute_sharpe_ratio_fixed(ports_valid, betas, adj_w) if ports_valid is not None else None

            # 选择最佳解 - 偏好有正则化的解
            best_beta, best_K, best_train_SR = find_best_solution_robust(
                betas, Ks, train_SR, valid_SR, preference="regularized"
            )

            best_valid_sr = valid_SR[np.nanargmax(valid_SR)] if valid_SR is not None else None

            result = {
                'lambda0': lam0,
                'lambda2': lam2,
                'train_SR': best_train_SR,
                'valid_SR': best_valid_sr,
                'portsN': int(best_K),
                'beta': best_beta
            }
            best_overall_results.append(result)
            
            print(f"  {lam0:7.1f} | {lam2:7.2f} | {best_K:6d} | {best_train_SR:6.4f} | {best_valid_sr if best_valid_sr is not None else 'N/A':6.4f}")

    if not best_overall_results:
        print("未找到有效解")
        return None

    # 选择最终最优解：优先选择有正则化的解,如果没有有正则化的解，才考虑无正则化的
    valid_results = [r for r in best_overall_results if r['lambda2'] > 0]
    if valid_results:
        best_idx = np.argmax([r.get('valid_SR', -np.inf) for r in valid_results])
        best_result = valid_results[best_idx]
        print(f"选择有正则化的解: lambda0={best_result['lambda0']}, lambda2={best_result['lambda2']}")
    else:
        best_idx = np.argmax([r.get('valid_SR', -np.inf) for r in best_overall_results])
        best_result = best_overall_results[best_idx]
        print(f"使用无正则化的解，可能过拟合")

    # 合并 train + valid 时间序列数据
    if ports_valid is not None and ports_train is not None:
        ports_trainval = np.vstack([ports_train, ports_valid])
    else:
        ports_trainval = ports_train.copy() if ports_train is not None else None

    # 用选定的 lambda0, lambda2 在 train+valid 上重新构造 sigma_tilde, mu_tilde 并求解
    try:
        sigma_tilde_tv, mu_tilde_tv, _ = prepare_sigma_mu_tilde_fixed(ports_trainval, [best_result['lambda0']])
        mu_tilde_tv = mu_tilde_tv.flatten()
        betas_tv, Ks_tv = lasso_fixed(sigma_tilde_tv, mu_tilde_tv, best_result['lambda2'], kmin=kmin, kmax=kmax)

        if len(betas_tv) == 0:
            print("在 train+valid 上重新训练未返回解，使用原训练集解作为最终解。")
            beta_final = best_result['beta'].copy()
            best_K_final = best_result['portsN']
        else:
            trainval_SR = compute_sharpe_ratio_fixed(ports_trainval, betas_tv, adj_w)
            idx_best_tv = int(np.nanargmax(trainval_SR))
            beta_final = betas_tv[idx_best_tv]
            best_K_final = int(Ks_tv[idx_best_tv])

            print(f"在 train+valid 上重新训练并选出最终解: K={best_K_final}, idx={idx_best_tv}")
    except Exception as e:
        print(f"在 train+valid 上重新训练时出错: {e}")
        beta_final = best_result['beta'].copy()
        best_K_final = best_result['portsN']

    # 用最终 beta_final 计算权重并继续后续流程
    beta_final = beta_final.copy()
    b = beta_final * adj_w
    weights = normalize_weights_china(b)
    best_result['weights_sum1'] = weights
    best_result['portsN'] = best_K_final


    # 测试集评估
    if ports_test is not None:
        test_port_returns = ports_test @ weights

        check_data_quality(test_port_returns)
        
        is_excess_return = config.get('is_excess_return', True)
        frequency = config.get('frequency', 'monthly')
        days_per_month = config.get('days_per_month', 30)
        
        performance_metrics = calculate_realistic_performance(
            test_port_returns, is_excess_return, frequency, days_per_month
        )
        
        best_result.update(performance_metrics)
        best_result['test_returns_series'] = test_port_returns
        
        print(f"\n 真实测试集性能报告（直接年化公式）:")
        print(f"  累计收益率: {performance_metrics['累计收益率']:.2%}")
        print(f"  投资天数: {performance_metrics['投资天数']}天")
        print(f"  年化收益率: {performance_metrics['年化收益率']:.2%}")
        print(f"  年化波动率: {performance_metrics['年化波动率']:.2%}")
        print(f"  年化夏普比率: {performance_metrics['年化夏普比率']:.3f}")
        print(f"  最大回撤: {performance_metrics['最大回撤']:.2%}")
        print(f"  Calmar比率: {performance_metrics['Calmar比率']:.3f}")
        print(f"  胜率: {performance_metrics['胜率']:.2%}")
        print(f"  偏度: {performance_metrics['偏度']:.3f}")
        print(f"  峰度: {performance_metrics['峰度']:.3f}")
        print(f"  计算方法: 直接年化公式 (年化收益率 = 累计收益 / 5)")

    out_best = {
        'lambda0': best_result['lambda0'],
        'lambda2': best_result['lambda2'],
        'portsN': best_result['portsN'],
        'train_SR': best_result['train_SR'],
        'valid_SR': best_result.get('valid_SR', np.nan),
        'total_return': best_result.get('累计收益率', np.nan),
        'investment_days': best_result.get('投资天数', np.nan),
        'annual_return': best_result.get('年化收益率', np.nan),
        'annual_volatility': best_result.get('年化波动率', np.nan),
        'annual_sharpe': best_result.get('年化夏普比率', np.nan),
        'max_drawdown': best_result.get('最大回撤', np.nan),
        'calmar_ratio': best_result.get('Calmar比率', np.nan),
        'win_rate': best_result.get('胜率', np.nan),
        'skewness': best_result.get('偏度', np.nan),
        'kurtosis': best_result.get('峰度', np.nan)
    }
    
    results_df = pd.DataFrame([out_best])
    results_df.to_csv(output_dir / 'best_results_realistic.csv', index=False)

    weights_df = pd.DataFrame({
        'node_index': np.arange(len(weights)),
        'weight': weights
    })
    weights_df.to_csv(output_dir / 'selected_weights_indexed.csv', index=False)

    if best_result['test_returns_series'] is not None:
        pd.DataFrame({'test_return': best_result['test_returns_series']}).to_csv(
            output_dir / 'test_returns_series.csv', index=False)

    return best_result

def run_ap_pruning_on_generated_files(returns_path, descriptions_path, config):
    print("开始改进版 AP-Pruning（中国市场适配）...")
    print(f"  returns: {returns_path}")
    print(f"  nodes:   {descriptions_path}")

    returns_df = pd.read_csv(returns_path)
    nodes_df = pd.read_csv(descriptions_path)

    if 'date' not in returns_df.columns:
        raise ValueError("returns file 必须包含 'date' 列")
    
    returns_df['date'] = pd.to_datetime(returns_df['date'], errors='coerce')
    returns_df = returns_df.sort_values('date').reset_index(drop=True)

    node_cols = [c for c in returns_df.columns if c != 'date']
    ports_data = returns_df[node_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    
    total_required = config['train_size'] + config['valid_size'] + config['test_size']
    if len(ports_data) < total_required:
        raise ValueError(f"数据长度不足")

    # 计算深度权重
    depth_map = nodes_df.set_index('node_name')['depth'].to_dict()
    depths = np.array([depth_map.get(col, 0) for col in node_cols], dtype=float)
    adj_w = 1.0 / np.sqrt(config['depth_weight_power'] ** depths)
    adj_w = np.nan_to_num(adj_w, nan=1.0, posinf=1.0, neginf=1.0)

    # 数据划分：每 5 期划分一次，前 4 期训练，第 5 期验证,测试集不变 
    T = len(ports_data)
    block_size = 5
    train_per_block = 4
    valid_per_block = 1

    train_idx = []
    valid_idx = []

    # 保留测试集为最后 test_size 个观测
    test_idx = np.arange(T - config['test_size'], T)

    # 剩余部分用于训练+验证
    tv_end = T - config['test_size']
    
    for start in range(0, tv_end, block_size):
        end = min(start + block_size, tv_end)
        block = np.arange(start, end)

        # 如果块不足 5 个也要处理
        if len(block) <= train_per_block:
            # 不足部分全部给训练
            train_idx.extend(block)
        else:
            train_idx.extend(block[:train_per_block])
            valid_idx.extend(block[train_per_block:train_per_block + valid_per_block])

    train_idx = np.array(train_idx)
    valid_idx = np.array(valid_idx)

    # 取数据
    ports_train = ports_data.iloc[train_idx].values
    ports_valid = ports_data.iloc[valid_idx].values
    ports_test = ports_data.iloc[test_idx].values

    print(f"新的划分：train={len(train_idx)}, valid={len(valid_idx)}, test={len(test_idx)}")

    print(f"节点数: {len(node_cols)}, 中国市场: 只做多归一化")
    print(f"收益率类型: {'超额收益' if config.get('is_excess_return', True) else '原始收益'}")
    print(f"每月天数假设: {config.get('days_per_month', 30)}天")

    # 运行改进的AP-Pruning
    main_dir = Path(config['output_dir'])
    best_result = lasso_pruning_helper(ports_train, ports_valid, ports_test,
                                       config['lambda0_list'], config['lambda2_list'],
                                       main_dir, adj_w, config['kmin'], config['kmax'], config)

    if best_result is not None:
        weights = best_result['weights_sum1']
        weights_named = pd.DataFrame({
            'node_name': node_cols,
            'weight': weights
        }).sort_values('weight', ascending=False).reset_index(drop=True)
        weights_named.to_csv(Path(config['output_dir']) / 'selected_weights_by_node.csv', index=False)
        
        positive_weights = weights[weights > 0]
        print(f"权重统计: 正权重{len(positive_weights)}个, 零权重{len(weights) - len(positive_weights)}个")
        print(f"权重总和: {np.sum(weights):.6f} (应为1.0)")
        
    return config['output_dir'], best_result

def run_ap_pruning_only():
    print("改进版 AP-Pruning（中国市场）+ 训练、验证集按4：1打乱")
    config = APPRUNING_CONFIG.copy()

    returns_path = input("请输入收益数据文件路径: ").strip()
    descriptions_path = input("请输入节点描述文件路径: ").strip()

    # 确认收益率类型
    is_excess = input("收益率是否已经是超额收益（减去无风险利率）? (y/n, 默认: y): ").strip().lower()
    if is_excess == 'n':
        config['is_excess_return'] = False
        print("注意: 将使用原始收益率计算夏普比率")
    else:
        config['is_excess_return'] = True
        print("使用超额收益率计算夏普比率")

    # 允许修改每月天数假设
    days_input = input(f"每月天数假设 (默认: {config.get('days_per_month', 30)}): ").strip()
    if days_input:
        config['days_per_month'] = int(days_input)

    if not os.path.exists(returns_path) or not os.path.exists(descriptions_path):
        print("输入文件不存在")
        return None

    output_dir, best_result = run_ap_pruning_on_generated_files(returns_path, descriptions_path, config)
    return output_dir, best_result

if __name__ == "__main__" and 'get_ipython' in globals():
    run_ap_pruning_only()